# Task 6.5: Coverage report

Explain what reached `merged_v0.csv`, what did not, and why. This notebook reads existing files and creates `coverage_report.md` in the project folder. It does not change the merged dataset or the earlier notebooks.

**Selected reporting rules**
- Keep the Task 6 base of 798 Eloundou jobs.
- An original occupation is represented if **at least one** of its mapped 2018 codes appears in v0. Show partially represented splits separately.
- Use project files only. The supplied Frey–Osborne CSV contains 653 of the paper's stated 702 jobs. Report the 49-job input gap without inventing their codes.
- A missing right-side match is not a dropped left-side job. Left joins retain all base jobs.
- Combining detailed jobs or averaging repeated mappings is not the same as dropping source occupations.

In [2]:
import pandas as pd
from pathlib import Path

project_folder = Path('..') if Path('../data/datasets').is_dir() else Path('.')
data_folder = project_folder / 'data/datasets'
processed_folder = data_folder / 'processed'

master = pd.read_csv(processed_folder / 'merged_v0.csv', dtype={'soc2018': 'string'})
eloundou = pd.read_csv(data_folder / 'eloundou_6digit.csv', dtype={'soc_6digit': 'string'})
raw_eloundou = pd.read_csv(data_folder / 'occ_level.csv', dtype={'O*NET-SOC Code': 'string'})
raw_eloundou['soc2018'] = raw_eloundou['O*NET-SOC Code'].str.split('.').str[0]

raw_aioe = pd.read_excel(data_folder / 'AIOE_appendixA.xlsx', dtype={'SOC Code': 'string'})
raw_frey = pd.read_csv(data_folder / 'frey_osborne_probabilities.csv', dtype={'soc_code_2010': 'string'})
aioe = pd.read_csv(processed_folder / 'aioe_2018soc.csv', dtype={'soc2010': 'string', 'soc2018': 'string'})
frey = pd.read_csv(processed_folder / 'frey_osborne_2018soc.csv', dtype={'soc2010': 'string', 'soc2018': 'string'})
crosswalk = pd.read_csv(processed_folder / 'crosswalk_clean.csv', dtype={'2010 SOC Code': 'string', '2018 SOC Code': 'string'})

onet_files = {
    'job_zones': 'onet_job_zones_6digit.csv',
    'abilities': 'onet_abilities_6digit.csv',
    'skills': 'onet_skills_6digit.csv',
    'work_activities': 'onet_work_activities_6digit.csv',
    'work_context': 'onet_work_context_6digit.csv',
}
onet = {}
for name, filename in onet_files.items():
    onet[name] = pd.read_csv(processed_folder / filename, dtype={'soc_6digit': 'string'}).rename(columns={'soc_6digit': 'soc2018'})

anchor_codes = set(master['soc2018'])
assert master['soc2018'].is_unique and master['soc2018'].notna().all()
assert anchor_codes == set(eloundou['soc_6digit']), 'v0 no longer matches the selected Eloundou base.'
print('v0:', len(master), 'jobs and', len(master.columns), 'columns')

v0: 798 jobs and 214 columns


## 1. Trace original occupations through the crosswalk

Count original 2010 codes separately from the 2018 codes they produce. A source job can be fully represented, partly represented, or have no contribution to v0. The input gap for Frey–Osborne is a separate fourth category.

In [3]:
lineage_rows = []
for source, table in [('AIOE', aioe), ('Frey–Osborne', frey)]:
    for old_code, group in table.groupby('soc2010'):
        mapped = set(group['soc2018'].dropna())
        retained = mapped & anchor_codes
        excluded = mapped - anchor_codes
        status = 'fully represented' if retained and not excluded else (
            'partly represented' if retained else 'not represented'
        )
        if not mapped:
            reason = ('Crosswalk granularity gap: 19-1020 Biologists has no exact mapping; '
                      'the supplied crosswalk lists detailed children 19-1021, 19-1022, 19-1023, and 19-1029.')
            if old_code != '19-1020':
                reason = 'No exact mapping in the supplied crosswalk; cause requires review.'
        elif excluded:
            reason = 'Mapped 2018 code(s) absent from the selected Eloundou base.'
        else:
            reason = 'All mapped 2018 codes appear in v0.'
        lineage_rows.append({
            'source': source, 'soc2010': old_code, 'original_title': group['title_2010'].iloc[0],
            'retained_2018_codes': '; '.join(sorted(retained)),
            'excluded_2018_codes': '; '.join(sorted(excluded)),
            'status': status, 'reason': reason,
        })

source_lineage = pd.DataFrame(lineage_rows)
assert set(source_lineage.loc[source_lineage['source'].eq('AIOE'), 'soc2010']) == set(raw_aioe['SOC Code'])
assert set(source_lineage.loc[source_lineage['source'].eq('Frey–Osborne'), 'soc2010']) == set(raw_frey['soc_code_2010'])

lost_originals = source_lineage[source_lineage['status'].eq('not represented')].copy()
partial_originals = source_lineage[source_lineage['status'].eq('partly represented')].copy()
source_lineage.groupby(['source', 'status']).size().unstack(fill_value=0)

status,fully represented,not represented,partly represented
source,,,
AIOE,763,5,6
Frey–Osborne,648,3,2


In [4]:
# Fractions use original occupations, not expanded crosswalk rows
original_coverage_rows = [{
    'Source': 'Eloundou', 'Original reference': 923, 'Available source jobs': len(raw_eloundou),
    'Represented source jobs': int(raw_eloundou['soc2018'].isin(anchor_codes).sum()),
    'Partly represented': 0, 'Available but not represented': 0, 'Unavailable at input': 0,
}]
for source, reference, available in [('AIOE', 774, len(raw_aioe)), ('Frey–Osborne', 702, len(raw_frey))]:
    rows = source_lineage[source_lineage['source'].eq(source)]
    original_coverage_rows.append({
        'Source': source, 'Original reference': reference, 'Available source jobs': available,
        'Represented source jobs': int(rows['status'].ne('not represented').sum()),
        'Partly represented': int(rows['status'].eq('partly represented').sum()),
        'Available but not represented': int(rows['status'].eq('not represented').sum()),
        'Unavailable at input': reference - available,
    })

original_coverage = pd.DataFrame(original_coverage_rows)
original_coverage['Share of original reference'] = (
    original_coverage['Represented source jobs'] / original_coverage['Original reference'] * 100
).map(lambda value: f'{value:.1f}%')
original_coverage['Share of available source'] = (
    original_coverage['Represented source jobs'] / original_coverage['Available source jobs'] * 100
).map(lambda value: f'{value:.1f}%')
assert (original_coverage['Represented source jobs'] + original_coverage['Available but not represented']
        + original_coverage['Unavailable at input']).equals(original_coverage['Original reference'])
original_coverage

,Source,Original reference,Available source jobs,Represented source jobs,Partly represented,Available but not represented,Unavailable at input,Share of original reference,Share of available source
0,Eloundou,923,923,923,0,0,0,100.0%,100.0%
1,AIOE,774,774,769,6,5,0,99.4%,99.4%
2,Frey–Osborne,702,653,650,2,3,49,92.6%,99.5%


## 2. Preparation and crosswalk accounting

Record the change in units before the final joins. The Eloundou 923-to-798 reduction is a rollup, not 125 lost jobs. AIOE and Frey–Osborne can expand during the crosswalk and then shrink when repeated target codes are averaged.

In [5]:
preparation_rows = []
for source, raw_count, table in [('AIOE', len(raw_aioe), aioe), ('Frey–Osborne', len(raw_frey), frey)]:
    matched = table[table['soc2018'].notna()]
    preparation_rows.append({
        'Source': source, 'Original rows in': raw_count,
        'Original rows matched': matched['soc2010'].nunique(),
        'Original rows without mapping': raw_count - matched['soc2010'].nunique(),
        'Rows after left crosswalk join': len(table),
        'Unmapped rows set aside': int(table['soc2018'].isna().sum()),
        'Mapped rows before averaging': len(matched),
        'Unique 2018 codes after averaging': matched['soc2018'].nunique(),
        'Unique codes outside v0': len(set(matched['soc2018']) - anchor_codes),
    })
preparation_funnel = pd.DataFrame(preparation_rows)
preparation_funnel

,Source,Original rows in,Original rows matched,Original rows without mapping,Rows after left crosswalk join,Unmapped rows set aside,Mapped rows before averaging,Unique 2018 codes after averaging,Unique codes outside v0
0,AIOE,774,773,1,827,1,826,800,10
1,Frey–Osborne,653,653,0,681,0,681,668,5


In [6]:
# O*NET files are long tables: repeated feature records are not separate jobs
raw_onet_specs = {
    'job_zones': ['onet_job_zones.xlsx'],
    'abilities': ['onet_abilities.xlsx'],
    'skills': ['onet_essential_skills.xlsx', 'onet_transferable_skills.xlsx'],
    'work_activities': ['onet_work_activities.xlsx'],
    'work_context': ['onet_work_context.xlsx'],
}
onet_inventory_rows = []
for name, filenames in raw_onet_specs.items():
    frames = [pd.read_excel(data_folder / filename, usecols=['O*NET-SOC Code']) for filename in filenames]
    raw_codes = pd.concat(frames, ignore_index=True)['O*NET-SOC Code'].astype('string')
    broad_codes = raw_codes.str.split('.').str[0]
    processed_codes = set(onet[name]['soc2018'])
    assert set(broad_codes) == processed_codes, f'{name}: source codes were lost during preparation.'
    onet_inventory_rows.append({
        'Source': name, 'Raw records': len(raw_codes), 'Detailed source jobs': raw_codes.nunique(),
        'Broad source jobs': broad_codes.nunique(), 'Broad jobs in v0': len(processed_codes & anchor_codes),
        'Source jobs outside v0': len(processed_codes - anchor_codes),
    })
onet_inventory = pd.DataFrame(onet_inventory_rows)
onet_inventory

,Source,Raw records,Detailed source jobs,Broad source jobs,Broad jobs in v0,Source jobs outside v0
0,job_zones,923,923,798,798,0
1,abilities,92976,894,774,774,0
2,skills,62580,894,774,774,0
3,work_activities,73308,894,774,774,0
4,work_context,297676,894,774,774,0


## 3. Main join funnel: every left-join step

Reconstruct Task 6's joins using unique code lists, in the same order. Matched counts refer to jobs on the left, not nonempty numeric scores. Rows dropped refers to the left side; right-side codes excluded by the base choice are shown separately.

In [7]:
source_keys = {
    'bls_titles': crosswalk[['2018 SOC Code']].rename(columns={'2018 SOC Code': 'soc2018'}).drop_duplicates(),
    'aioe': aioe[['soc2018']].dropna().drop_duplicates(),
    'frey': frey[['soc2018']].dropna().drop_duplicates(),
}
for name, table in onet.items():
    source_keys[name] = table[['soc2018']].copy()

spine = eloundou[['soc_6digit']].rename(columns={'soc_6digit': 'soc2018'})
funnel_rows = []
for name, right in source_keys.items():
    rows_in = len(spine)
    joined = spine.merge(right, on='soc2018', how='left', indicator=True, validate='one_to_one')
    matched = int(joined['_merge'].eq('both').sum())
    funnel_rows.append({
        'Join step': name, 'Rows in': rows_in, 'Right-side codes': len(right),
        'Rows matched': matched, 'Unmatched rows kept': rows_in - matched,
        'Rows dropped': rows_in - len(joined), 'Rows out': len(joined),
        'Right-side codes outside base': len(set(right['soc2018']) - anchor_codes),
    })
    if name != 'bls_titles':
        expected_flags = master['soc2018'].isin(set(right['soc2018']))
        assert expected_flags.equals(master['in_' + name]), f'{name}: saved presence flags disagree with source codes.'
    spine = joined[['soc2018']]
    print(f'{name}: {rows_in} in, {matched} matched, {rows_in - len(joined)} dropped, {len(joined)} out')

join_funnel = pd.DataFrame(funnel_rows)
assert set(spine['soc2018']) == anchor_codes
join_funnel

bls_titles: 798 in, 798 matched, 0 dropped, 798 out
aioe: 798 in, 790 matched, 0 dropped, 798 out
frey: 798 in, 663 matched, 0 dropped, 798 out
job_zones: 798 in, 798 matched, 0 dropped, 798 out
abilities: 798 in, 774 matched, 0 dropped, 798 out
skills: 798 in, 774 matched, 0 dropped, 798 out
work_activities: 798 in, 774 matched, 0 dropped, 798 out
work_context: 798 in, 774 matched, 0 dropped, 798 out


,Join step,Rows in,Right-side codes,Rows matched,Unmatched rows kept,Rows dropped,Rows out,Right-side codes outside base
0,bls_titles,798,867,798,0,0,798,69
1,aioe,798,800,790,8,0,798,10
2,frey,798,668,663,135,0,798,5
3,job_zones,798,798,798,0,0,798,0
4,abilities,798,774,774,24,0,798,0
5,skills,798,774,774,24,0,798,0
6,work_activities,798,774,774,24,0,798,0
7,work_context,798,774,774,24,0,798,0


## 4. Full outer coverage audit and the final flag join

The outer audit starts with the Eloundou codes and adds any source-only codes. This is a separate coverage table; it does not replace the 798-row deliverable. Task 6 then left-joins the presence flags back into v0, which is included as the final step in the main funnel.

In [8]:
audit = eloundou[['soc_6digit']].rename(columns={'soc_6digit': 'soc2018'})
outer_rows = []
for name, right in source_keys.items():
    if name == 'bls_titles':
        continue
    rows_in = len(audit)
    joined = audit.merge(right, on='soc2018', how='outer', indicator=True, validate='one_to_one')
    outer_rows.append({
        'Source added': name, 'Rows in': rows_in,
        'Rows matched': int(joined['_merge'].eq('both').sum()),
        'Existing unmatched rows kept': int(joined['_merge'].eq('left_only').sum()),
        'New source-only codes': int(joined['_merge'].eq('right_only').sum()),
        'Rows dropped': 0, 'Rows out': len(joined),
    })
    audit = joined[['soc2018']]

outer_funnel = pd.DataFrame(outer_rows)
flag_join = spine.merge(audit, on='soc2018', how='left', indicator=True, validate='one_to_one')
join_funnel.loc[len(join_funnel)] = {
    'Join step': 'coverage flags', 'Rows in': len(spine), 'Right-side codes': len(audit),
    'Rows matched': int(flag_join['_merge'].eq('both').sum()),
    'Unmatched rows kept': int(flag_join['_merge'].eq('left_only').sum()),
    'Rows dropped': len(spine) - len(flag_join), 'Rows out': len(flag_join),
    'Right-side codes outside base': len(set(audit['soc2018']) - anchor_codes),
}
outside_codes = set(audit['soc2018']) - anchor_codes
outer_funnel

,Source added,Rows in,Rows matched,Existing unmatched rows kept,New source-only codes,Rows dropped,Rows out
0,aioe,798,790,8,10,0,808
1,frey,808,668,140,0,0,808
2,job_zones,808,798,10,0,0,808
3,abilities,808,774,34,0,0,808
4,skills,808,774,34,0,0,808
5,work_activities,808,774,34,0,0,808
6,work_context,808,774,34,0,0,808


## 5. Unmatched-code appendix

Keep three different cases separate: original source jobs contributing nothing, partly retained splits, and newer codes excluded by the base choice. Also list jobs retained in v0 that lack one or more sources. Describe what the files demonstrate; do not label a code discontinued or a typo without evidence.

In [9]:
official_titles = crosswalk[['2018 SOC Code', '2018 SOC Title']].drop_duplicates().set_index('2018 SOC Code')['2018 SOC Title']
excluded_rows = []
for soc in sorted(outside_codes):
    sources = []
    old_codes = set()
    for name, table in [('AIOE', aioe), ('Frey–Osborne', frey)]:
        matches = table[table['soc2018'].eq(soc)]
        if len(matches):
            sources.append(name)
            old_codes.update(matches['soc2010'])
    excluded_rows.append({
        '2018 SOC': soc, 'Official title': official_titles.loc[soc],
        'Sources': '; '.join(sources), 'Contributing 2010 codes': '; '.join(sorted(old_codes)),
        'Reason': 'Valid 2018 code absent from the selected Eloundou base; excluded by the left-join choice.',
    })
excluded_codes = pd.DataFrame(excluded_rows)
excluded_codes

,2018 SOC,Official title,Sources,Contributing 2010 codes,Reason
0,11-9039,"Education Administrators, All Other",AIOE,11-9039,Valid 2018 code absent from the selected Eloun...
1,25-3099,"Teachers and Instructors, All Other",AIOE,25-3099,Valid 2018 code absent from the selected Eloun...
2,25-9049,"Teaching Assistants, All Other",AIOE; Frey–Osborne,25-9041,Valid 2018 code absent from the selected Eloun...
3,27-3099,"Media and Communication Workers, All Other",AIOE; Frey–Osborne,27-3012,Valid 2018 code absent from the selected Eloun...
4,29-1249,"Surgeons, All Other",AIOE,29-1067,Valid 2018 code absent from the selected Eloun...
5,43-2099,"Communications Equipment Operators, All Other",AIOE; Frey–Osborne,27-4013,Valid 2018 code absent from the selected Eloun...
6,47-5049,"Underground Mining Machine Operators, All Other",AIOE; Frey–Osborne,47-5042,Valid 2018 code absent from the selected Eloun...
7,51-9199,"Production Workers, All Other",AIOE,51-9199,Valid 2018 code absent from the selected Eloun...
8,53-1049,First Line Supervisors of Transportation Worke...,AIOE,53-1031,Valid 2018 code absent from the selected Eloun...
9,53-7199,"Material Moving Workers, All Other",AIOE; Frey–Osborne,53-7032,Valid 2018 code absent from the selected Eloun...


In [10]:
# One row per retained job with missing source coverage
gap_rows = []
for row in master.itertuples(index=False):
    missing = [name for name in ['aioe', 'frey', *onet] if not getattr(row, 'in_' + name)]
    if not missing:
        continue
    explanations = []
    for name, available_codes in [('aioe', set(raw_aioe['SOC Code'])), ('frey', set(raw_frey['soc_code_2010']))]:
        if name in missing:
            old = set(crosswalk.loc[crosswalk['2018 SOC Code'].eq(row.soc2018), '2010 SOC Code'])
            if old and not (old & available_codes):
                explanations.append(name + ': mapped 2010 predecessor(s) absent from supplied index: ' + ', '.join(sorted(old)))
            else:
                explanations.append(name + ': source/crosswalk coverage gap requiring review')
    missing_onet = [name for name in onet if name in missing]
    if missing_onet:
        explanations.append('O*NET: this broad code is absent from the supplied rating files; the upstream cause is not established.')
    gap_rows.append({
        '2018 SOC': row.soc2018, 'Official title': row.title,
        'Missing sources': '; '.join(missing), 'Evidence / likely reason': ' | '.join(explanations),
    })
kept_source_gaps = pd.DataFrame(gap_rows)

presence_columns = [column for column in master if column.startswith('in_')]
base_coverage = pd.DataFrame({
    'Source': [column.removeprefix('in_') for column in presence_columns],
    'Jobs with source': [int(master[column].sum()) for column in presence_columns],
    'Jobs without source': [int((~master[column]).sum()) for column in presence_columns],
})
base_coverage['Share of v0'] = (base_coverage['Jobs with source'] / len(master) * 100).map(lambda value: f'{value:.1f}%')
base_coverage

,Source,Jobs with source,Jobs without source,Share of v0
0,eloundou,798,0,100.0%
1,aioe,790,8,99.0%
2,frey,663,135,83.1%
3,job_zones,798,0,100.0%
4,abilities,774,24,97.0%
5,skills,774,24,97.0%
6,work_activities,774,24,97.0%
7,work_context,774,24,97.0%


## 6. Check that the counts reconcile

These checks verify the report's source accounting and compare reconstructed joins with the saved v0. They do not rerun or change Task 6's chosen score calculations.

In [11]:
# Check that the final table retains the contributing original index codes counted above
for source, column in [('AIOE', 'aioe_source_codes'), ('Frey–Osborne', 'fo_source_codes')]:
    final_sources = set(master[column].dropna().str.split(';').explode())
    expected_sources = set(source_lineage.loc[source_lineage['source'].eq(source) & source_lineage['status'].ne('not represented'), 'soc2010'])
    assert final_sources == expected_sources, f'{source}: lineage counts differ from the final table.'

all_source_jobs = int(master[presence_columns].all(axis=1).sum())
all_index_jobs = int(master[['aioe', 'fo_prob', 'dv_rating_beta', 'human_rating_beta']].notna().all(axis=1).sum())
title_gap_codes = set(master.loc[master['eloundou_title'].isna(), 'soc2018'])
feature_gap_codes = set(master.loc[~master['in_abilities'], 'soc2018'])
print('Main-table jobs:', len(master))
print('Codes across all sources:', len(audit))
print('Jobs with all index scores:', all_index_jobs)
print('Jobs present in every prepared source:', all_source_jobs)
print('Excluded 2018 codes:', len(excluded_codes))
print('Original source jobs with no contribution:', len(lost_originals))
print('Partly represented original source jobs:', len(partial_originals))
print('Retained jobs missing one or more sources:', len(kept_source_gaps))
print('Overlap between former title gaps and O*NET coverage gaps:', len(title_gap_codes & feature_gap_codes))

Main-table jobs: 798
Codes across all sources: 808
Jobs with all index scores: 663
Jobs present in every prepared source: 650
Excluded 2018 codes: 10
Original source jobs with no contribution: 8
Partly represented original source jobs: 8
Retained jobs missing one or more sources: 148
Overlap between former title gaps and O*NET coverage gaps: 0
